# 01 - v1 GLWM, corrected re-run (motivation and tooling check)
Re-implements the v1 GLWM (GRU encoder, residual latent loop, PonderNet-style gate, halting at
gate > 0.7, T_max = 8) on 1-4 digit addition, with the fixes the review asked for:
the gate initialisation as published (`v1`: both gate biases -2) and as intended (`fixed`),
a validation split for model selection, a test set de-duplicated against training, 5 seeds,
and FLOPs measured with `FlopCounterMode` instead of hand-derived formulas.

In [1]:
# parameters
SEEDS = [0, 1, 2, 3, 4]
INITS = ["v1", "fixed"]
EPOCHS = 30

In [2]:
%matplotlib inline
import json, sys, time
from pathlib import Path
import numpy as np, pandas as pd, torch
import matplotlib.pyplot as plt
from glwm2 import PROJECT_ROOT, RESULTS_DIR, PAPER_DIR
from glwm2.eval.plotting import set_style
set_style()
WRITTEN = []  # files this notebook writes

In [3]:
from glwm2.models.v1_glwm import train_v1
from glwm2.utils.io import read_json, write_json
OUT = RESULTS_DIR / "v1"
res = []
for init in INITS:
    for seed in SEEDS:
        p = OUT / f"{init}_s{seed}.json"
        if not p.exists():
            r, _ = train_v1(init, seed, epochs=EPOCHS)
            write_json(p, r)
        res.append(read_json(p)); WRITTEN.append(p)
print(len(res), "runs")

W1013 09:20:20.258000 3152 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


10 runs


In [4]:
rows = []
for r in res:
    for proto in ["test_v1_protocol", "test_dedup"]:
        t = r[proto]
        rows.append(dict(init=r["gate_init"], seed=r["seed"], protocol=proto, acc=t["acc"], mean_steps=t["mean_steps"],
                         steps_4digit=t["steps_by_digits"]["4"] if "4" in t["steps_by_digits"] else t["steps_by_digits"][4],
                         acc_step1=t["acc_at_step"][0], gflops_per_sample=r["flops"]["per_sample_at_mean_steps"] / 1e9))
df = pd.DataFrame(rows)
summary = df.groupby(["init", "protocol"]).agg(["mean", "std"]).drop(columns="seed")
summary

acc      mean_steps           steps_4digit            \
                       mean  std       mean       std         mean       std   
init  protocol                                                                 
fixed test_dedup        1.0  0.0    3.40702  0.121068     3.867372  0.177246   
      test_v1_protocol  1.0  0.0    2.47294  0.070972     3.868720  0.177355   
v1    test_dedup        1.0  0.0    2.59638  0.077788     3.052949  0.056224   
      test_v1_protocol  1.0  0.0    2.02728  0.043194     3.055530  0.053937   

                       acc_step1           gflops_per_sample            
                            mean       std              mean       std  
init  protocol                                                          
fixed test_dedup         0.58880  0.034880          0.007499  0.000071  
      test_v1_protocol   0.78086  0.022201          0.007499  0.000071  
v1    test_dedup         0.84048  0.044853          0.007025  0.000046  
      test_v1_protocol   0.91778  0.024790          0.007025  0.000046

In [5]:
dup = pd.DataFrame([r["test_dup_share_by_digits"] for r in res]).mean()
print("share of v1-protocol test pairs that also appear in train/val, by digit count:")
print(dup.round(3).to_string())

share of v1-protocol test pairs that also appear in train/val, by digit count:
1    1.000
2    0.966
3    0.034
4    0.000


In [6]:
from glwm2.eval.plotting import savefig
fig, axes = plt.subplots(1, 2, figsize=(6.8, 2.4))
for init, c in zip(INITS, ["#D55E00", "#009E73"]):
    H = np.mean([r["test_dedup"]["steps_hist"] for r in res if r["gate_init"] == init], 0)
    axes[0].bar(np.arange(1, 9) + (0.2 if init == "fixed" else -0.2), H / H.sum(), width=0.4, color=c, label=f"gate init: {init}")
    A = np.mean([r["test_dedup"]["acc_at_step"] for r in res if r["gate_init"] == init], 0)
    axes[1].plot(np.arange(1, 9), A, "o-", color=c, label=init)
axes[0].set_xlabel("halting step"); axes[0].set_ylabel("share of test instances"); axes[0].legend(frameon=False)
axes[1].set_xlabel("latent step t"); axes[1].set_ylabel("exact-match accuracy at step t"); axes[1].set_ylim(0.9, 1.001)
WRITTEN.append(savefig(fig, "figA_v1_corrected"))

In [7]:
tab = df[df.protocol == "test_dedup"].groupby("init").agg(
    acc=("acc", "mean"), acc_sd=("acc", "std"), steps=("mean_steps", "mean"), steps4=("steps_4digit", "mean"),
    acc1=("acc_step1", "mean"), gflops=("gflops_per_sample", "mean"))
lines = [r"\begin{tabular}{lrrrrr}", r"\toprule",
         r"Gate init & Accuracy & Mean steps & Steps (4-digit) & Acc.\ at step 1 & MFLOPs/sample \\", r"\midrule"]
for init, r in tab.iterrows():
    lines.append(f"{init} & {r.acc:.4f} $\\pm$ {r.acc_sd:.4f} & {r.steps:.2f} & {r.steps4:.2f} & {r.acc1:.4f} & {r.gflops*1e3:.2f} \\\\")
lines += [r"\bottomrule", r"\end{tabular}"]
p = PAPER_DIR / "tables" / "v1_corrected.tex"; p.write_text("\n".join(lines)); WRITTEN.append(p)
print("\n".join(lines))
WRITTEN.append(write_json(RESULTS_DIR / "v1" / "summary.json", tab.reset_index().to_dict(orient="records")))

\begin{tabular}{lrrrrr}
\toprule
Gate init & Accuracy & Mean steps & Steps (4-digit) & Acc.\ at step 1 & MFLOPs/sample \\
\midrule
fixed & 1.0000 $\pm$ 0.0000 & 3.41 & 3.87 & 0.5888 & 7.50 \\
v1 & 1.0000 $\pm$ 0.0000 & 2.60 & 3.05 & 0.8405 & 7.02 \\
\bottomrule
\end{tabular}


In [8]:
print("Files written by this notebook:")
for p in WRITTEN:
    print(" ", Path(p).relative_to(PROJECT_ROOT))

Files written by this notebook:
  results\v1\v1_s0.json
  results\v1\v1_s1.json
  results\v1\v1_s2.json
  results\v1\v1_s3.json
  results\v1\v1_s4.json
  results\v1\fixed_s0.json
  results\v1\fixed_s1.json
  results\v1\fixed_s2.json
  results\v1\fixed_s3.json
  results\v1\fixed_s4.json
  paper\figures\figA_v1_corrected.pdf
  paper\tables\v1_corrected.tex
  results\v1\summary.json
